In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath = projectPath / 'pipelines' / 'operational' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
gapCases = pl.read_parquet(dataPrepPath / 'opticalGapCases.parquet').filter(pl.col('interpolationEvaluable'))
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
comparisonRows = predictions.join(gapCases.select('rowId', 'ndvi', 'interpolationNdvi', 'bracketingGapDays', 'interpolationGapRange'), on='rowId', how='inner').with_columns((pl.col('prediction') - pl.col('ndvi')).abs().alias('radarAbsError'), (pl.col('interpolationNdvi') - pl.col('ndvi')).abs().alias('interpolationAbsError')).with_columns((pl.col('radarAbsError') - pl.col('interpolationAbsError')).alias('errorDifference'), (pl.col('radarAbsError') < pl.col('interpolationAbsError')).alias('radarWins'))
radarVsInterpolation = comparisonRows.group_by('model', 'featureSet', 'seed', 'interpolationGapRange').agg(pl.col('bracketingGapDays').min().alias('minimumGapDays'), pl.col('bracketingGapDays').max().alias('maximumGapDays'), pl.col('radarAbsError').mean().alias('radarMae'), pl.col('interpolationAbsError').mean().alias('interpolationMae'), pl.col('errorDifference').mean().alias('meanErrorDifference'), pl.col('radarWins').mean().alias('radarWinRate'), pl.len().alias('rowCount')).sort('model', 'featureSet', 'seed', 'minimumGapDays')
radarVsInterpolation.write_parquet(outputPath / 'radarVsInterpolation.parquet')
assert comparisonRows.select(pl.col('rowId').n_unique()).item() == 716
assert radarVsInterpolation.select(pl.col('rowCount').sum()).item() == comparisonRows.height
radarVsInterpolation
